# B2-022-probabilistic-latent-models — Practice p04 — Solution

**Type:** mc · **Difficulty:** intro · **Concepts:** autoencoder

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

Shape ledger for a batch of $B=32$:

| stage | module | output shape |
|---|---|---|
| input `x` | — | `(32, 20)` |
| encoder | `Linear(20, 8)` + ReLU | `(32, 8)` |
| **bottleneck (code)** | `Linear(8, 3)` | **`(32, 3)`** |
| decoder | `Linear(3, 8)` + ReLU | `(32, 8)` |
| reconstruction `x_hat` | `Linear(8, 20)` | `(32, 20)` |

A `Linear` layer acts on the last axis only, so the batch axis 32 is preserved at every stage.
An autoencoder is trained to reproduce its input, so the reconstruction target is `x` itself, shape `(32, 20)` (no labels are involved).
The code width $3<20$, so this autoencoder is **undercomplete**.

**Answer: B.**

In [ ]:
import torch
from torch import nn

torch.manual_seed(20260927)
encoder = nn.Sequential(nn.Linear(20, 8), nn.ReLU(), nn.Linear(8, 3))
decoder = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 20))
x = torch.randn(32, 20)
ledger = [("input", tuple(x.shape))]
h = x
for name, module in zip(("enc_linear", "enc_relu", "code"), encoder):
    h = module(h)
    ledger.append((name, tuple(h.shape)))
code = h
for name, module in zip(("dec_linear", "dec_relu", "reconstruction"), decoder):
    h = module(h)
    ledger.append((name, tuple(h.shape)))
x_hat = h
loss = nn.functional.mse_loss(x_hat, x)   # target is x itself
ANSWER = "B"
print(ledger)

### Answer check

In [ ]:
# Checks: the autoencoder answers against pinned values
assert tuple(code.shape) == (32, 3)
assert x_hat.shape == x.shape == (32, 20)
assert x.dtype == torch.float32 and x_hat.dtype == torch.float32
assert code.shape[1] < x.shape[1]       # undercomplete
assert loss.ndim == 0
assert ANSWER == "B"